# First step of the project
Training Resnet on CIFAR10 dataset (expected accuracy 90%)
https://medium.com/@thatchawin.ler/cifar10-with-resnet-in-pytorch-a86fe18049df

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import random_split, DataLoader

try:
  from torchinfo import summary
except:
  !pip install torchinfo
  from torchinfo import summary
import torchvision
from torchvision import datasets
from torchvision import transforms
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import pandas as pd
import seaborn as sn
import numpy as np
import random
from typing import List , Dict , Tuple
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
device = torch.device('cuda')
print(device)

cuda


In [4]:
dataset = torchvision.datasets.CIFAR10(
    root='/content/',
    train=True,
    download=True,
    transform=data_transform
)

train_ratio = 0.8
val_ratio = 0.2
train_size = int(train_ratio * len(dataset))
val_size = len(dataset) - train_size

trainset, valset = random_split(dataset, [train_size, val_size])

testset = torchvision.datasets.CIFAR10(
    root='/content/',
    train=False,
    download=True,
    transform=data_transform
)

classes = dataset.classes
print(classes)

100%|██████████| 170M/170M [00:13<00:00, 12.2MB/s]


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [5]:
NUM_WORKERS = os.cpu_count()
BATCH_SIZE = 800

train_dataloader = DataLoader(dataset=trainset,
                              batch_size=BATCH_SIZE,
                              num_workers=NUM_WORKERS,
                              shuffle=True)

test_dataloader = DataLoader(dataset=testset,
                             batch_size=BATCH_SIZE,
                             num_workers=NUM_WORKERS,
                             shuffle=False)

val_dataloader = DataLoader(dataset=valset,
                            batch_size=BATCH_SIZE,
                            num_workers=NUM_WORKERS,
                            shuffle=False)

def train_step(model:nn.Module,
               dataloader:torch.utils.data.DataLoader,
               loss_fn:nn.Module,
               optimizer:torch.optim.Optimizer,
               scheduler:torch.optim.lr_scheduler = None,
               grad_clip:float=None):

  model.train()

  train_loss = 0
  train_acc = 0

  for batch, (X,y) in enumerate(dataloader):
    X = X.to(device)
    y = y.to(device)
    y_pred = model(X)
    loss = loss_fn(y_pred,y)
    train_loss += loss.item()
    optimizer.zero_grad()
    loss.backward()

    if grad_clip:
      nn.utils.clip_grad_value_(model.parameters(), grad_clip)

    optimizer.step()
    y_pred_class = torch.argmax(y_pred,dim=1)
    train_acc += (y_pred_class == y).sum().item() / len(y)

  train_loss /= len(dataloader)
  train_acc /= len(dataloader)

  if scheduler is not None:
    scheduler.step(train_loss)


  return train_loss , train_acc

def test_step(model:nn.Module,
              dataloader:torch.utils.data.DataLoader,
              loss_fn:nn.Module):

  model.eval()

  test_loss = 0
  test_acc = 0

  with torch.inference_mode():
    for batch, (X,y) in enumerate(dataloader):
      X = X.to(device)
      y = y.to(device)
      test_pred_logits = model(X)
      loss = loss_fn(test_pred_logits,y)
      test_loss += loss.item()
      test_pred_labels = torch.argmax(test_pred_logits,dim=1)
      test_acc += (test_pred_labels == y).sum().item() / len(y)

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)

  return test_loss , test_acc

In [6]:
from tqdm.auto import tqdm
#Patryk: train przerobiłem, żeby zapisywało najlepszą z epok do pliku
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          scheduler:torch.optim.lr_scheduler,
          grad_clip:float=None,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 10):

    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }
    best_val_loss = float('inf')

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           scheduler=scheduler,
                                           grad_clip=grad_clip)

        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn)

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        if test_loss < best_val_loss:
            best_val_loss = test_loss
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"\n New best model saved. Val Loss: {best_val_loss:.4f}\n")


    return results


In [7]:
def plot_loss_curves(results: Dict[str, List[float]]):

    loss = results['train_loss']
    test_loss = results['test_loss']

    accuracy = results['train_acc']
    test_accuracy = results['test_acc']

    epochs = range(len(results['train_loss']))

    plt.figure(figsize=(15, 7))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label='train_loss')
    plt.plot(epochs, test_loss, label='test_loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.grid()
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label='train_accuracy')
    plt.plot(epochs, test_accuracy, label='test_accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.grid()
    plt.legend()

In [8]:
def plot_loss_curves(results: Dict[str, List[float]]):
    loss = results['train_loss']
    test_loss = results['test_loss']

    accuracy = results['train_acc']
    test_accuracy = results['test_acc']

    epochs = range(len(results['train_loss']))

    plt.figure(figsize=(15, 7))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label='train_loss')
    plt.plot(epochs, test_loss, label='test_loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.grid()
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label='train_accuracy')
    plt.plot(epochs, test_accuracy, label='test_accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.grid()
    plt.legend()

In [9]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += self.shortcut(x)
        out = self.relu(out)
        return out

class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.initial_conv = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64, 64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(64, 128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, num_blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, num_blocks=2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn(self.initial_conv(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

model_resnet18 = ResNet18(num_classes=10).to(device)
summary(model_resnet18, input_size=[1, 3, 32, 32])

Layer (type:depth-idx)                   Output Shape              Param #
ResNet18                                 [1, 10]                   --
├─Conv2d: 1-1                            [1, 64, 32, 32]           1,728
├─BatchNorm2d: 1-2                       [1, 64, 32, 32]           128
├─ReLU: 1-3                              [1, 64, 32, 32]           --
├─Sequential: 1-4                        [1, 64, 32, 32]           --
│    └─BasicBlock: 2-1                   [1, 64, 32, 32]           --
│    │    └─Conv2d: 3-1                  [1, 64, 32, 32]           36,864
│    │    └─BatchNorm2d: 3-2             [1, 64, 32, 32]           128
│    │    └─ReLU: 3-3                    [1, 64, 32, 32]           --
│    │    └─Conv2d: 3-4                  [1, 64, 32, 32]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 32, 32]           128
│    │    └─Sequential: 3-6              [1, 64, 32, 32]           --
│    │    └─ReLU: 3-7                    [1, 64, 32, 32]           --
│

In [10]:
model_2 = ResNet18(num_classes=10).to(device)
summary(model_2, input_size=[1, 3, 32, 32])

Layer (type:depth-idx)                   Output Shape              Param #
ResNet18                                 [1, 10]                   --
├─Conv2d: 1-1                            [1, 64, 32, 32]           1,728
├─BatchNorm2d: 1-2                       [1, 64, 32, 32]           128
├─ReLU: 1-3                              [1, 64, 32, 32]           --
├─Sequential: 1-4                        [1, 64, 32, 32]           --
│    └─BasicBlock: 2-1                   [1, 64, 32, 32]           --
│    │    └─Conv2d: 3-1                  [1, 64, 32, 32]           36,864
│    │    └─BatchNorm2d: 3-2             [1, 64, 32, 32]           128
│    │    └─ReLU: 3-3                    [1, 64, 32, 32]           --
│    │    └─Conv2d: 3-4                  [1, 64, 32, 32]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 32, 32]           128
│    │    └─Sequential: 3-6              [1, 64, 32, 32]           --
│    │    └─ReLU: 3-7                    [1, 64, 32, 32]           --
│

In [13]:
NUM_EPOCHS = 30
learning_rate = 0.001

weight_decay = 35e-5
grad_clip = 0.0001

SAVE_DIR = '/content/drive/MyDrive/model_base/'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'resnet18_cifar10_baseline.pth')
print(f"Model saved in: {MODEL_SAVE_PATH}")

model_2 = ResNet18(num_classes=10).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_2.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='min', factor=0.3, patience=3, threshold=0.09)

from timeit import default_timer as timer
start_time = timer()

model_2_results = train(model=model_2,
                        train_dataloader=train_dataloader,
                        test_dataloader=val_dataloader,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        grad_clip=grad_clip,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS)

end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

Model saved in: /content/drive/MyDrive/model_base/resnet18_cifar10_baseline.pth


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 1.7423 | train_acc: 0.3729 | test_loss: 1.6896 | test_acc: 0.4109

 New best model saved. Val Loss: 1.6896

Epoch: 2 | train_loss: 1.2978 | train_acc: 0.5285 | test_loss: 1.4298 | test_acc: 0.4970

 New best model saved. Val Loss: 1.4298

Epoch: 3 | train_loss: 1.1306 | train_acc: 0.5949 | test_loss: 1.3684 | test_acc: 0.5209

 New best model saved. Val Loss: 1.3684

Epoch: 4 | train_loss: 1.0329 | train_acc: 0.6325 | test_loss: 1.3894 | test_acc: 0.5236
Epoch: 5 | train_loss: 0.9262 | train_acc: 0.6685 | test_loss: 1.1416 | test_acc: 0.5903

 New best model saved. Val Loss: 1.1416

Epoch: 6 | train_loss: 0.8528 | train_acc: 0.7016 | test_loss: 1.4069 | test_acc: 0.5397
Epoch: 7 | train_loss: 0.7986 | train_acc: 0.7199 | test_loss: 1.2251 | test_acc: 0.5705
Epoch: 8 | train_loss: 0.7760 | train_acc: 0.7284 | test_loss: 1.2835 | test_acc: 0.5819
Epoch: 9 | train_loss: 0.7353 | train_acc: 0.7458 | test_loss: 1.0932 | test_acc: 0.6228

 New best model saved. Val Los

In [14]:
plot_loss_curves(model_2_results)

In [15]:
predicted_labels = []
actual_labels = []

model_2.eval()

with torch.no_grad():
  for images, labels in DataLoader(dataset=testset, batch_size=1, num_workers=NUM_WORKERS):
    images, labels = images.to(device), labels.to(device)
    prediction_logits = model_2(images)
    predictions = prediction_logits.argmax(dim=1).cpu().numpy()
    predicted_labels.extend(predictions)
    true_labels = labels.cpu().numpy()
    actual_labels.extend(true_labels)

confusion_mat = confusion_matrix(actual_labels, predicted_labels)
confusion_df = pd.DataFrame(confusion_mat/np.sum(confusion_mat)*10, index=classes, columns=classes)
plt.figure(figsize=(12,7))
sn.heatmap(confusion_df, annot=True)
plt.show()

<Figure size 1200x700 with 0 Axes>

<Figure size 1200x700 with 0 Axes>

In [16]:
import gc

del model_2
del optimizer
del scheduler
del model_2_results


torch.cuda.empty_cache()

gc.collect()

print("Memory released.")
print(f"Current GPU Mem Usage: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

Memory released.
Current GPU Mem Usage: 0.10 GB


In [21]:
class ContrastiveTransformations:
    def __init__(self, base_transforms):
        self.base_transforms = base_transforms

    def __call__(self, x):
        q = self.base_transforms(x)
        k = self.base_transforms(x)
        return [q, k]

mean, std = [0.4914, 0.4822, 0.4465], [0.247, 0.243, 0.261] # średnia i odchylenie standardowe dla pikseli w CIFAR-10

contrastive_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

print("Tworzenie Contrastive Dataset...")
train_contrastive_dataset = torchvision.datasets.CIFAR10(
    root='/content/',
    train=True,
    download=True,
    transform=ContrastiveTransformations(contrastive_transforms)
)

contrastive_dataloader = DataLoader(
    dataset=train_contrastive_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True,
    drop_last=True
)

Tworzenie Contrastive Dataset...


In [22]:
class SimCLR(nn.Module):
    def __init__(self, base_encoder, projection_dim=128):
        super().__init__()
        self.encoder = base_encoder
        if hasattr(self.encoder, 'fc'):
            num_features = self.encoder.fc.in_features
            self.encoder.fc = nn.Identity()
        elif hasattr(self.encoder, 'classifier'):
            num_features = self.encoder.classifier[-1].in_features
            self.encoder.classifier = nn.Sequential(
                nn.MaxPool2d(4),
                nn.Flatten()
            )
        else:
            raise ValueError("Nieznana architektura enkodera (brak fc lub classifier)")

        self.projection_head = nn.Sequential(
            nn.Linear(num_features, num_features, bias=False),
            nn.ReLU(),
            nn.Linear(num_features, projection_dim, bias=False)
        )

    def forward(self, x):
        features = self.encoder(x)
        projections = self.projection_head(features)
        return projections

class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()
        self.similarity_f = nn.CosineSimilarity(dim=2)

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        z = torch.cat([z_i, z_j], dim=0)
        sim_matrix = self.similarity_f(z.unsqueeze(1), z.unsqueeze(0))
        sim_ij = torch.diag(sim_matrix, batch_size)
        sim_ji = torch.diag(sim_matrix, -batch_size)
        positive_samples = torch.cat([sim_ij, sim_ji], dim=0).reshape(2 * batch_size, 1)

        mask = (~torch.eye(2 * batch_size, 2 * batch_size, dtype=bool)).to(device)

        negative_samples = sim_matrix[mask].reshape(2 * batch_size, -1)
        logits = torch.cat([positive_samples, negative_samples], dim=1)
        logits /= self.temperature
        labels = torch.zeros(2 * batch_size).to(device).long()
        loss = self.criterion(logits, labels)
        return loss

In [23]:
PRETRAIN_EPOCHS = 100
PRETRAIN_LR = 1e-3 #zwiekszylem, bo w paperze jest inny optymalizator(lars) i w paperze pisali, ze lr powinien byc dla simclr wysoki
WEIGHT_DECAY = 1e-4

base_encoder = ResNet18(num_classes=10)
pretrain_model = SimCLR(base_encoder=base_encoder).to(device)

pretrain_loss_fn = NTXentLoss()
pretrain_optimizer = torch.optim.AdamW(pretrain_model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
ENCODER_SAVE_PATH = os.path.join(SAVE_DIR, 'simclr_resnet18_encoder.pth')
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    pretrain_optimizer,
    T_max=PRETRAIN_EPOCHS,
    eta_min=1e-6
)
print("Pre-training")

for epoch in tqdm(range(PRETRAIN_EPOCHS)):
    pretrain_model.train()
    total_loss = 0

    for batch in contrastive_dataloader:
        images, _ = batch
        images_i, images_j = images[0].to(device), images[1].to(device)

        proj_i = pretrain_model(images_i)
        proj_j = pretrain_model(images_j)

        loss = pretrain_loss_fn(proj_i, proj_j)
        total_loss += loss.item()

        pretrain_optimizer.zero_grad()
        loss.backward()
        pretrain_optimizer.step()

    scheduler.step()

    if (epoch+1) % 10 == 0:
        avg_loss = total_loss / len(contrastive_dataloader)
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch: {epoch+1:03d} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

torch.save(pretrain_model.encoder.state_dict(), ENCODER_SAVE_PATH)
print(f"\nZakończono pre-training. Wagi enkodera zapisane w: {ENCODER_SAVE_PATH}")

Starting Pre-training SimCLR with ResNet18...


  0%|          | 0/100 [00:00<?, ?it/s]

Pre-train Epoch: 001/100 | Loss: 6.5310
Pre-train Epoch: 002/100 | Loss: 6.2374
Pre-train Epoch: 003/100 | Loss: 6.1311
Pre-train Epoch: 004/100 | Loss: 6.0666
Pre-train Epoch: 005/100 | Loss: 6.0193
Pre-train Epoch: 006/100 | Loss: 5.9869
Pre-train Epoch: 007/100 | Loss: 5.9509
Pre-train Epoch: 008/100 | Loss: 5.9289
Pre-train Epoch: 009/100 | Loss: 5.9087
Pre-train Epoch: 010/100 | Loss: 5.8908
Pre-train Epoch: 011/100 | Loss: 5.8820
Pre-train Epoch: 012/100 | Loss: 5.8677
Pre-train Epoch: 013/100 | Loss: 5.8614
Pre-train Epoch: 014/100 | Loss: 5.8478
Pre-train Epoch: 015/100 | Loss: 5.8452
Pre-train Epoch: 016/100 | Loss: 5.8382
Pre-train Epoch: 017/100 | Loss: 5.8292
Pre-train Epoch: 018/100 | Loss: 5.8229
Pre-train Epoch: 019/100 | Loss: 5.8191
Pre-train Epoch: 020/100 | Loss: 5.8136
Pre-train Epoch: 021/100 | Loss: 5.8138
Pre-train Epoch: 022/100 | Loss: 5.8047
Pre-train Epoch: 023/100 | Loss: 5.8014
Pre-train Epoch: 024/100 | Loss: 5.7993
Pre-train Epoch: 025/100 | Loss: 5.7932


In [24]:
import gc

del pretrain_model
del pretrain_optimizer
del contrastive_dataloader

torch.cuda.empty_cache()

gc.collect()

print("Memory released.")
print(f"Current GPU Mem Usage: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

Memory released.
Current GPU Mem Usage: 0.25 GB


In [25]:
finetune_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

full_train_dataset = torchvision.datasets.CIFAR10(
    root='/content/',
    train=True,
    download=True,
    transform=finetune_transforms
)


targets = np.array(full_train_dataset.targets)
num_classes = len(full_train_dataset.classes)
finetune_indices = []

for i in range(num_classes):
    class_indices = np.where(targets == i)[0]
    num_samples = min(50, len(class_indices))
    finetune_indices.extend(np.random.choice(class_indices, num_samples, replace=False))

finetune_dataset = torch.utils.data.Subset(full_train_dataset, finetune_indices)

print(f"Created fine-tunning dataset from {len(finetune_dataset)} images ({len(finetune_dataset) / len(full_train_dataset) * 100:.1f}% of all).")

finetune_dataloader = DataLoader(
    dataset=finetune_dataset,
    batch_size=64,
    num_workers=NUM_WORKERS,
    shuffle=True
)

Created fine-tunning dataset from 500 images (1.0% of all).


In [26]:
finetune_model = ResNet18(num_classes=10).to(device)

ENCODER_SAVE_PATH = os.path.join(SAVE_DIR, 'simclr_resnet18_encoder.pth')

finetune_model.fc = nn.Identity()
finetune_model.load_state_dict(torch.load(ENCODER_SAVE_PATH))

for param in finetune_model.parameters():
    param.requires_grad = False

finetune_model.fc = nn.Linear(512, 10).to(device)

print("Parametry, które będą trenowane:")
for name, param in finetune_model.named_parameters():
    if param.requires_grad:
        print(f"   -> {name}")

FINETUNE_EPOCHS = 20
FINETUNE_LR = 1e-3
FINETUNE_SAVE_PATH = os.path.join(SAVE_DIR, 'resnet18_finetuned_1_percent.pth')
MODEL_SAVE_PATH = FINETUNE_SAVE_PATH

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(finetune_model.fc.parameters(), lr=FINETUNE_LR)

finetune_results = train(model=finetune_model,
                         train_dataloader=finetune_dataloader,
                         test_dataloader=val_dataloader,
                         optimizer=optimizer,
                         scheduler=None,
                         grad_clip=None,
                         loss_fn=loss_fn,
                         epochs=FINETUNE_EPOCHS)

print(f"\n Fine-tuning zakończony. Najlepszy model w: {FINETUNE_SAVE_PATH}")

Parametry, które będą trenowane:
   -> fc.weight
   -> fc.bias


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 2.0828 | train_acc: 0.2994 | test_loss: 1.8747 | test_acc: 0.3807

 New best model saved. Val Loss: 1.8747

Epoch: 2 | train_loss: 1.4320 | train_acc: 0.5957 | test_loss: 1.5462 | test_acc: 0.5472

 New best model saved. Val Loss: 1.5462

Epoch: 3 | train_loss: 1.0664 | train_acc: 0.7360 | test_loss: 1.3983 | test_acc: 0.5750

 New best model saved. Val Loss: 1.3983

Epoch: 4 | train_loss: 0.9126 | train_acc: 0.7258 | test_loss: 1.3146 | test_acc: 0.5860

 New best model saved. Val Loss: 1.3146

Epoch: 5 | train_loss: 0.8180 | train_acc: 0.7613 | test_loss: 1.2633 | test_acc: 0.5906

 New best model saved. Val Loss: 1.2633

Epoch: 6 | train_loss: 0.7381 | train_acc: 0.7870 | test_loss: 1.2377 | test_acc: 0.5920

 New best model saved. Val Loss: 1.2377

Epoch: 7 | train_loss: 0.7057 | train_acc: 0.7895 | test_loss: 1.1961 | test_acc: 0.6069

 New best model saved. Val Loss: 1.1961

Epoch: 8 | train_loss: 0.6819 | train_acc: 0.7919 | test_loss: 1.1952 | test_acc: 0

In [27]:
# ewaluacja

final_model = ResNet18(num_classes=10).to(device)
final_model.load_state_dict(torch.load(FINETUNE_SAVE_PATH))

test_loss_1_percent, test_acc_1_percent = test_step(
    model=final_model,
    dataloader=test_dataloader,
    loss_fn=nn.CrossEntropyLoss()
)

print("\n--- WYNIKI KOŃCOWE (ResNet18) ---")
print(f"Dokładność modelu SimCLR po fine-tuningu na 1% danych: {test_acc_1_percent*100:.2f}%")
classical_model_path = os.path.join(SAVE_DIR, 'resnet18_cifar10_baseline.pth')
classical_model = ResNet18(num_classes=10).to(device)
classical_model.load_state_dict(torch.load(classical_model_path))

_, test_acc_100_percent = test_step(
    model=classical_model,
    dataloader=test_dataloader,
    loss_fn=nn.CrossEntropyLoss()
)
print(f"Dokładność modelu klasycznego (100% danych): {test_acc_100_percent*100:.2f}%")
print("------------------------")


--- WYNIKI KOŃCOWE (ResNet18) ---
Dokładność modelu SimCLR po fine-tuningu na 1% danych: 59.94%
Dokładność modelu klasycznego (100% danych): 85.44%
------------------------
